# Extra experiments
A: cold-start thermal drift · B: duty-cycle (power mechanism) · C: 128×128 robustness · D: NCCL transport test

**Run with Accelerator = GPU T4 x2, via Save & Run All. Experiment A must run first, on a cool GPU.**

In [1]:
# ---- Cell 1: config and helpers ----
import os, time, json, csv, subprocess, sys, socket, importlib
import torch, torch.nn as nn
import pandas as pd

RESULTS_DIR = '/kaggle/working/results_extra'
os.makedirs(RESULTS_DIR, exist_ok=True)

MODELS = ['resnet18', 'resnet34', 'resnet50', 'vittiny', 'vitbase']
NUM_CLASSES = 200
WARMUP, MEASURED = 20, 100
WARM_SEC = 60
D0, D1 = 'cuda:0', 'cuda:1'

def run_path(model, config, bs, res):
    return f"{RESULTS_DIR}/{model}-{config}-bs{bs}-res{res}.csv"

def already_done(path):
    if os.path.exists(path):
        print("skip (exists):", os.path.basename(path))
        return True
    return False

def write_csv(path, header, rows):
    tmp = path + '.tmp'
    with open(tmp, 'w', newline='') as fh:
        w = csv.writer(fh); w.writerow(header); w.writerows(rows)
    os.replace(tmp, path)

def gpu_temp(i=0):
    out = subprocess.run(['nvidia-smi', '--query-gpu=temperature.gpu',
                          '--format=csv,noheader,nounits', '-i', str(i)],
                         capture_output=True, text=True).stdout
    return int(out.strip())


In [2]:
# ---- Cell 2: environment (also records the starting GPU temperature) ----
def nccl_version():
    try:
        return str(torch.cuda.nccl.version())
    except Exception as e:
        return f"unavailable ({type(e).__name__})"

COLD_TEMP = gpu_temp(0)
info = {
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'nccl': nccl_version(),
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    'p2p_0_to_1': torch.cuda.device_count() > 1 and torch.cuda.can_device_access_peer(0, 1),
    'gpu0_temp_at_start_c': COLD_TEMP,
    'gpu1_temp_at_start_c': gpu_temp(1) if torch.cuda.device_count() > 1 else None,
    'topology': subprocess.run(['nvidia-smi', 'topo', '-m'], capture_output=True, text=True).stdout,
}
json.dump(info, open(f'{RESULTS_DIR}/environment.json', 'w'), indent=2)
print(json.dumps(info, indent=2))
assert len(info['gpus']) == 2, "Need 2 GPUs: set Accelerator to GPU T4 x2"


{
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "nccl": "(2, 28, 9)",
  "gpus": [
    "Tesla T4",
    "Tesla T4"
  ],
  "p2p_0_to_1": true,
  "gpu0_temp_at_start_c": 52,
  "gpu1_temp_at_start_c": 54,
  "topology": "\t\u001bGPU0\tGPU1\tCPU Affinity\tNUMA Affinity\tGPU NUMA ID\u001b\nGPU0\t X \tPHB\t0-3\t0\t\tN/A\nGPU1\tPHB\t X \t0-3\t0\t\tN/A\n\nLegend:\n\n  X    = Self\n  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)\n  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node\n  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)\n  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)\n  PIX  = Connection traversing at most a single PCIe bridge\n  NV#  = Connection traversing a bonded set of # NVLinks\n"
}


In [3]:
# ---- Cell 3: models and synthetic data ----
from torchvision import models
from torchvision.models.vision_transformer import VisionTransformer

def build_model(name, res):
    if name == 'resnet18': return models.resnet18(weights=None, num_classes=NUM_CLASSES)
    if name == 'resnet34': return models.resnet34(weights=None, num_classes=NUM_CLASSES)
    if name == 'resnet50': return models.resnet50(weights=None, num_classes=NUM_CLASSES)
    if name == 'vitbase':  return models.vit_b_16(weights=None, image_size=res, num_classes=NUM_CLASSES)
    if name == 'vittiny':  return VisionTransformer(image_size=res, patch_size=16, num_layers=12,
                                                    num_heads=3, hidden_dim=192, mlp_dim=768,
                                                    num_classes=NUM_CLASSES)
    raise ValueError(name)

def make_batch(bs, device, res, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(bs, 3, res, res, generator=g).to(device)
    y = torch.randint(0, NUM_CLASSES, (bs,), generator=g).to(device)
    return x, y


In [4]:
# ---- Cell 4: clock logger and cool-down helper ----
def start_clock_log(tag):
    f = open(f"{RESULTS_DIR}/{tag}-clocks.csv", 'w')
    p = subprocess.Popen(
        ['nvidia-smi', '--query-gpu=timestamp,index,clocks.sm,power.draw,temperature.gpu',
         '--format=csv,noheader,nounits', '-lms', '200'], stdout=f)
    return p, f

def stop_clock_log(p, f):
    p.terminate(); p.wait(); f.close()

def cooldown(target_c, max_min=12):
    t = time.time()
    while time.time() - t < max_min * 60 and gpu_temp(0) > target_c:
        time.sleep(15)
    print(f"cool-down: GPU0 at {gpu_temp(0)}°C after {(time.time() - t) / 60:.1f} min (target {target_c}°C)")


## Experiment A: cold-start thermal drift
Starts on a cool GPU with **no warm-up**, runs continuously for 8 minutes, and records every step. Shows how step time drifts as the GPU heats up, and how long a warm-up must really be.

In [5]:
# ---- Cell 5: Experiment A ----
def cold_start(model_name, bs, res=224, seconds=480):
    path = run_path(model_name, 'coldstart', bs, res)
    if already_done(path): return
    model = build_model(model_name, res).to(D0).train()
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    x, y = make_batch(bs, D0, res)
    temp0 = gpu_temp(0)
    tag = os.path.basename(path)[:-4]
    p, f = start_clock_log(tag)
    rows, t0, i = [], time.perf_counter(), 0
    while time.perf_counter() - t0 < seconds:
        torch.cuda.synchronize(D0); s = time.perf_counter()
        opt.zero_grad(set_to_none=True)
        crit(model(x), y).backward()
        opt.step()
        torch.cuda.synchronize(D0); e = time.perf_counter()
        rows.append([i, round(time.time(), 3), round(s - t0, 3), round((e - s) * 1e3, 3)])
        i += 1
    stop_clock_log(p, f)
    write_csv(path, ['step', 'wall', 't_s', 'step_ms'], rows)
    early = pd.Series([r[3] for r in rows[20:120]]).median()
    late = pd.Series([r[3] for r in rows[-100:]]).median()
    print(f"{tag}: {temp0}°C -> {gpu_temp(0)}°C | first-100 median {early:.1f} ms | "
          f"last-100 median {late:.1f} ms ({100 * (late / early - 1):+.1f}%) | {i} steps")
    del model, opt, x, y
    torch.cuda.empty_cache()

cold_start('resnet18', 32)
cooldown(COLD_TEMP + 3)
cold_start('resnet50', 32)


resnet18-coldstart-bs32-res224: 53°C -> 75°C | first-100 median 89.9 ms | last-100 median 100.7 ms (+12.1%) | 4793 steps
cool-down: GPU0 at 66°C after 12.0 min (target 55°C)
resnet50-coldstart-bs32-res224: 66°C -> 77°C | first-100 median 363.8 ms | last-100 median 348.9 ms (-4.1%) | 1369 steps


## Pre-heat
Brings **both** GPUs to a steady temperature before Experiments B and C, so no run starts cool.

In [6]:
# ---- Cell 6: session pre-heat (both GPUs, 4 minutes) ----
def preheat(seconds=240):
    ms = [build_model('resnet50', 224).to(d).train() for d in (D0, D1)]
    xs = [make_batch(32, d, 224)[0] for d in (D0, D1)]
    t = time.time()
    while time.time() - t < seconds:
        for m, x in zip(ms, xs):
            m(x).sum().backward()          # both GPUs work at the same time (asynchronous launches)
        torch.cuda.synchronize(D0); torch.cuda.synchronize(D1)
    print(f"pre-heat done: GPU0 {gpu_temp(0)}°C, GPU1 {gpu_temp(1)}°C")
    del ms, xs
    torch.cuda.empty_cache()

preheat()


pre-heat done: GPU0 77°C, GPU1 70°C


## Experiment B: duty cycle (tests the power-limit mechanism)
One GPU runs the normal full training step, then sits idle for a gap of 0×, 0.5× or 1× its own step time. If busy time falls as the gap grows, idle time alone makes the GPU faster — the mechanism behind naive MP's speedup.

In [7]:
# ---- Cell 7: Experiment B ----
def duty_cycle(model_name, bs, ratio, res=224, warm_sec=90):
    path = run_path(model_name, f'duty{int(round(ratio * 100))}', bs, res)
    if already_done(path): return
    model = build_model(model_name, res).to(D0).train()
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    x, y = make_batch(bs, D0, res)

    def busy():
        torch.cuda.synchronize(D0); s = time.perf_counter()
        opt.zero_grad(set_to_none=True)
        crit(model(x), y).backward()
        opt.step()
        torch.cuda.synchronize(D0)
        return (time.perf_counter() - s) * 1e3

    for _ in range(10): busy()
    est = sorted(busy() for _ in range(11))[5]           # typical busy time
    gap = ratio * est / 1e3                               # idle seconds after each step

    t_w, n = time.perf_counter(), 0                       # warm up AT this duty cycle
    while n < WARMUP or time.perf_counter() - t_w < warm_sec:
        busy(); time.sleep(gap); n += 1

    tag = os.path.basename(path)[:-4]
    p, f = start_clock_log(tag)
    rows = []
    for i in range(MEASURED):
        b = busy()
        rows.append([model_name, bs, res, ratio, i, round(b, 3), round(gap * 1e3, 3)])
        time.sleep(gap)
    stop_clock_log(p, f)
    write_csv(path, ['model', 'batch', 'res', 'gap_ratio', 'step', 'busy_ms', 'gap_ms'], rows)
    med = pd.Series([r[5] for r in rows]).median()
    print(f"{tag}: busy {med:.1f} ms | gap {gap * 1e3:.1f} ms | GPU0 {gpu_temp(0)}°C")
    del model, opt, x, y
    torch.cuda.empty_cache()

DUTY_CASES = [('resnet18', 32), ('resnet50', 32), ('vittiny', 32), ('vitbase', 16)]
for m, bs in DUTY_CASES:
    for r in [0.0, 0.5, 1.0]:
        duty_cycle(m, bs, r)


resnet18-duty0-bs32-res224: busy 100.8 ms | gap 0.0 ms | GPU0 76°C
resnet18-duty50-bs32-res224: busy 90.6 ms | gap 49.5 ms | GPU0 75°C
resnet18-duty100-bs32-res224: busy 91.9 ms | gap 100.0 ms | GPU0 76°C
resnet50-duty0-bs32-res224: busy 354.0 ms | gap 0.0 ms | GPU0 75°C
resnet50-duty50-bs32-res224: busy 341.1 ms | gap 172.3 ms | GPU0 76°C
resnet50-duty100-bs32-res224: busy 347.4 ms | gap 358.4 ms | GPU0 75°C
vittiny-duty0-bs32-res224: busy 138.5 ms | gap 0.0 ms | GPU0 76°C
vittiny-duty50-bs32-res224: busy 125.6 ms | gap 69.0 ms | GPU0 75°C
vittiny-duty100-bs32-res224: busy 126.9 ms | gap 137.1 ms | GPU0 77°C
vitbase-duty0-bs16-res224: busy 619.9 ms | gap 0.0 ms | GPU0 76°C
vitbase-duty50-bs16-res224: busy 611.3 ms | gap 312.5 ms | GPU0 75°C
vitbase-duty100-bs16-res224: busy 612.7 ms | gap 635.4 ms | GPU0 76°C


## Experiment C: 128×128 robustness
All five models at 128×128: 1-GPU (batch 16 and 32), DP and MP (global batch 32), with the same protocol as the main runs.

In [8]:
# ---- Cell 8: 1-GPU benchmark (resolution as a parameter) ----
def bench_1gpu(model_name, bs, res, device=D0):
    path = run_path(model_name, '1gpu', bs, res)
    if already_done(path): return
    torch.cuda.empty_cache()
    model = build_model(model_name, res).to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    x, y = make_batch(bs, device, res)

    def fwd_bwd():
        opt.zero_grad(set_to_none=True)
        crit(model(x), y).backward()

    t_w, n = time.perf_counter(), 0
    while n < WARMUP or time.perf_counter() - t_w < WARM_SEC:
        fwd_bwd(); opt.step(); n += 1
    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)

    tag = os.path.basename(path)[:-4]
    p, f = start_clock_log(tag)
    rows = []
    for i in range(MEASURED):
        torch.cuda.synchronize(device); t0 = time.perf_counter()
        fwd_bwd()
        torch.cuda.synchronize(device); t1 = time.perf_counter()
        opt.step()
        torch.cuda.synchronize(device); t2 = time.perf_counter()
        rows.append([model_name, '1gpu', bs, res, 1, i,
                     round((t1 - t0) * 1e3, 3), round((t2 - t1) * 1e3, 3), round((t2 - t0) * 1e3, 3)])
    stop_clock_log(p, f)
    peak = torch.cuda.max_memory_allocated(device) / 2**20
    write_csv(path, ['model', 'config', 'batch', 'res', 'repeat', 'step',
                     'fwdbwd_ms', 'opt_ms', 'total_ms', 'peak_mem_mb'],
              [r + [round(peak, 1)] for r in rows])
    med = pd.Series([r[8] for r in rows]).median()
    print(f"{tag}: median {med:.1f} ms | peak {peak:.0f} MB | warm-up {n} steps")
    del model, opt, x, y
    torch.cuda.empty_cache()


In [9]:
# ---- Cell 9: model-parallel split, verified at 128x128, and MP benchmark ----
class ViTFront(nn.Module):
    def __init__(s, v):
        super().__init__()
        s.conv_proj, s.class_token = v.conv_proj, v.class_token
        s.pos, s.drop = v.encoder.pos_embedding, v.encoder.dropout
        s.blocks = nn.Sequential(*list(v.encoder.layers)[:6])
        s.h = v.hidden_dim
    def forward(s, x):
        n = x.shape[0]
        x = s.conv_proj(x).reshape(n, s.h, -1).permute(0, 2, 1)
        x = torch.cat([s.class_token.expand(n, -1, -1), x], dim=1) + s.pos
        return s.blocks(s.drop(x))

class ViTBack(nn.Module):
    def __init__(s, v):
        super().__init__()
        s.blocks = nn.Sequential(*list(v.encoder.layers)[6:])
        s.ln, s.heads = v.encoder.ln, v.heads
    def forward(s, x):
        return s.heads(s.ln(s.blocks(x))[:, 0])

def split(m, name):
    if name.startswith('resnet'):
        p1 = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool, m.layer1, m.layer2)
        p2 = nn.Sequential(m.layer3, m.layer4, m.avgpool, nn.Flatten(1), m.fc)
    else:
        p1, p2 = ViTFront(m), ViTBack(m)
    return p1, p2

for name in MODELS:                                   # verify the split at 128x128
    m = build_model(name, 128).to(D0).eval()
    p1, p2 = split(m, name)
    xx, _ = make_batch(4, D0, 128)
    with torch.no_grad():
        same = torch.allclose(m(xx), p2(p1(xx)), atol=1e-4)
    n_full = sum(p.numel() for p in m.parameters())
    n_split = sum(p.numel() for p in p1.parameters()) + sum(p.numel() for p in p2.parameters())
    print(f"{name:9s} (128) same output: {same} | all params: {n_full == n_split}")
    del m, p1, p2
torch.cuda.empty_cache()

def sync_both():
    torch.cuda.synchronize(D0); torch.cuda.synchronize(D1)

def bench_mp(model_name, bs, res):
    path = run_path(model_name, 'mp', bs, res)
    if already_done(path): return
    torch.cuda.empty_cache()
    p1, p2 = split(build_model(model_name, res), model_name)
    p1, p2 = p1.to(D0).train(), p2.to(D1).train()
    opt = torch.optim.SGD(list(p1.parameters()) + list(p2.parameters()), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    x, y = make_batch(bs, D0, res); y = y.to(D1)

    def natural():
        opt.zero_grad(set_to_none=True)
        crit(p2(p1(x).to(D1)), y).backward()
        opt.step()

    def natural_timed():
        sync_both(); t0 = time.perf_counter()
        natural()
        torch.cuda.synchronize(D0); ta = time.perf_counter()
        torch.cuda.synchronize(D1); tb = time.perf_counter()
        return (tb - t0) * 1e3, (ta - t0) * 1e3

    def staged():
        opt.zero_grad(set_to_none=True)
        t = []
        def mark(): sync_both(); t.append(time.perf_counter())
        mark()
        a = p1(x);                                mark()
        b = a.detach().to(D1).requires_grad_();   mark()
        loss = crit(p2(b), y);                    mark()
        loss.backward();                          mark()
        g = b.grad.to(D0);                        mark()
        a.backward(g);                            mark()
        opt.step();                               mark()
        return [(t[i + 1] - t[i]) * 1e3 for i in range(7)], a.numel() * 4 / 2**20

    t_w, n = time.perf_counter(), 0
    while n < WARMUP or time.perf_counter() - t_w < WARM_SEC:
        natural(); n += 1
    sync_both()
    for d in (D0, D1): torch.cuda.reset_peak_memory_stats(d)

    tag = os.path.basename(path)[:-4]
    p, f = start_clock_log(tag)
    rows = []
    for i in range(MEASURED):
        if i % 2 == 0:
            nat, nat0 = natural_timed(); ph, act_mb = staged()
        else:
            ph, act_mb = staged(); nat, nat0 = natural_timed()
        rows.append([model_name, 'mp', bs, res, 1, i, i % 2]
                    + [round(v, 3) for v in ph] + [round(sum(ph), 3), round(nat, 3), round(nat0, 3)])
    stop_clock_log(p, f)
    pk0, pk1 = (torch.cuda.max_memory_allocated(d) / 2**20 for d in (D0, D1))
    write_csv(path, ['model', 'config', 'batch', 'res', 'repeat', 'step', 'order',
                     'fwd0_ms', 'xfer_fwd_ms', 'fwd1_ms', 'bwd1_ms', 'xfer_bwd_ms', 'bwd0_ms',
                     'opt_ms', 'staged_total_ms', 'step_ms', 'step_sync0_ms',
                     'act_mb', 'peak0_mb', 'peak1_mb'],
              [r + [round(act_mb, 2), round(pk0, 1), round(pk1, 1)] for r in rows])
    d = pd.DataFrame(rows).iloc[:, 7:].median()
    print(f"{tag}: step {d.iloc[8]:.1f} ms | GPU0 {d.iloc[0] + d.iloc[5]:.1f} | "
          f"GPU1 {d.iloc[2] + d.iloc[3]:.1f} | xfer {d.iloc[1] + d.iloc[4]:.2f} ms | warm-up {n} steps")
    del p1, p2, opt, x, y
    torch.cuda.empty_cache()


resnet18  (128) same output: True | all params: True
resnet34  (128) same output: True | all params: True
resnet50  (128) same output: True | all params: True
vittiny   (128) same output: True | all params: True
vitbase   (128) same output: True | all params: True


In [10]:
%%writefile /kaggle/working/dp_bench.py
# ---- Cell 10: DP worker and all-reduce worker (run in separate processes) ----
import os, time, csv
from datetime import timedelta
import torch, torch.nn as nn, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.profiler import profile, ProfilerActivity
from torchvision import models
from torchvision.models.vision_transformer import VisionTransformer

NUM_CLASSES = 200

def build_model(name, res):
    if name == 'resnet18': return models.resnet18(weights=None, num_classes=NUM_CLASSES)
    if name == 'resnet34': return models.resnet34(weights=None, num_classes=NUM_CLASSES)
    if name == 'resnet50': return models.resnet50(weights=None, num_classes=NUM_CLASSES)
    if name == 'vitbase':  return models.vit_b_16(weights=None, image_size=res, num_classes=NUM_CLASSES)
    if name == 'vittiny':  return VisionTransformer(image_size=res, patch_size=16, num_layers=12,
                                                    num_heads=3, hidden_dim=192, mlp_dim=768,
                                                    num_classes=NUM_CLASSES)
    raise ValueError(name)

def make_batch(bs, device, res, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(bs, 3, res, res, generator=g).to(device)
    y = torch.randint(0, NUM_CLASSES, (bs,), generator=g).to(device)
    return x, y

def dp_worker(rank, world, model_name, global_bs, res, repeat, warmup, measured, path, port, warm_sec):
    os.environ['MASTER_ADDR'], os.environ['MASTER_PORT'] = 'localhost', str(port)
    dist.init_process_group('nccl', rank=rank, world_size=world, timeout=timedelta(minutes=10))
    torch.cuda.set_device(rank)
    dev, bs = f'cuda:{rank}', global_bs // world

    model = DDP(build_model(model_name, res).to(dev), device_ids=[rank])
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    crit = nn.CrossEntropyLoss()
    x, y = make_batch(bs, dev, res, seed=rank)

    def nosync():
        opt.zero_grad(set_to_none=True)
        with model.no_sync():
            crit(model(x), y).backward()
    def synced():
        opt.zero_grad(set_to_none=True)
        crit(model(x), y).backward()
    def timed(fn):
        torch.cuda.synchronize(dev); t = time.perf_counter()
        fn(); torch.cuda.synchronize(dev)
        return (time.perf_counter() - t) * 1e3

    t_w, n = time.perf_counter(), 0
    while True:
        nosync(); synced(); opt.step(); n += 1
        done = torch.tensor([float(n >= warmup and time.perf_counter() - t_w >= warm_sec)], device=dev)
        dist.broadcast(done, src=0)
        if done.item(): break
    torch.cuda.synchronize(dev)
    torch.cuda.reset_peak_memory_stats(dev)

    rows = []
    for i in range(measured):
        if i % 2 == 0:
            t_ns = timed(nosync); t_s = timed(synced); t_o = timed(opt.step)
        else:
            t_s = timed(synced); t_o = timed(opt.step); t_ns = timed(nosync)
        rows.append([model_name, 'dp', global_bs, bs, res, repeat, i, i % 2,
                     round(t_ns, 3), round(t_s, 3), round(t_o, 3),
                     round(t_s + t_o, 3), round(t_s - t_ns, 3)])

    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        for _ in range(20):
            synced(); opt.step()
        torch.cuda.synchronize(dev)
    dt = lambda e: getattr(e, 'device_time_total', 0) or getattr(e, 'cuda_time_total', 0)
    nccl = [e for e in prof.key_averages()
            if e.key.lower().startswith('nccl') and 'kernel' in e.key.lower()]
    nccl_ms = sum(dt(e) for e in nccl) / 1e3 / 20

    if rank == 0:
        peak = torch.cuda.max_memory_allocated(dev) / 2**20
        tmp = path + '.tmp'
        with open(tmp, 'w', newline='') as fh:
            w = csv.writer(fh)
            w.writerow(['model', 'config', 'batch', 'per_gpu_batch', 'res', 'repeat', 'step',
                        'order', 'nosync_ms', 'sync_ms', 'opt_ms', 'step_ms',
                        'exposed_comm_ms', 'nccl_ms', 'peak_mem_mb'])
            for r in rows: w.writerow(r + [round(nccl_ms, 3), round(peak, 1)])
        os.replace(tmp, path)
        med = lambda k: sorted(r[k] for r in rows)[measured // 2]
        print(f"{os.path.basename(path)[:-4]}: step {med(11):.1f} ms | "
              f"exposed {med(12):.1f} ms | nccl {nccl_ms:.1f} ms | warm-up {n} steps")
    dist.destroy_process_group()

def allreduce_worker(rank, world, sizes_mb, iters, path, port, extra_env):
    os.environ.update(extra_env)
    os.environ['NCCL_DEBUG'], os.environ['NCCL_DEBUG_SUBSYS'] = 'INFO', 'INIT'
    os.environ['MASTER_ADDR'], os.environ['MASTER_PORT'] = 'localhost', str(port)
    dist.init_process_group('nccl', rank=rank, world_size=world, timeout=timedelta(minutes=5))
    torch.cuda.set_device(rank)
    dev = f'cuda:{rank}'
    rows = []
    for mb in sizes_mb:
        t = torch.ones(int(mb * 2**20 / 4), device=dev)
        for _ in range(5): dist.all_reduce(t)
        torch.cuda.synchronize(dev)
        times = []
        for _ in range(iters):
            dist.barrier(device_ids=[rank]); torch.cuda.synchronize(dev)
            s = time.perf_counter()
            dist.all_reduce(t); torch.cuda.synchronize(dev)
            times.append((time.perf_counter() - s) * 1e3)
        med = sorted(times)[iters // 2]
        rows.append([round(mb, 2), round(med, 3), round(mb / 1024 / (med / 1e3), 3)])
    if rank == 0:
        with open(path, 'w', newline='') as fh:
            w = csv.writer(fh); w.writerow(['size_mb', 'median_ms', 'gb_per_s']); w.writerows(rows)
        for r in rows: print(f"all-reduce {r[0]:8.2f} MB: {r[1]:8.3f} ms  ->  {r[2]:.2f} GB/s")
    dist.destroy_process_group()


Writing /kaggle/working/dp_bench.py


In [11]:
# ---- Cell 11: DP launcher ----
sys.path.append('/kaggle/working')
import dp_bench; importlib.reload(dp_bench)
import torch.multiprocessing as mp

def free_port():
    s = socket.socket(); s.bind(('', 0)); port = s.getsockname()[1]; s.close()
    return port

def run_dp(m, bs, res):
    path = run_path(m, 'dp', bs, res)
    if already_done(path): return
    p, f = start_clock_log(os.path.basename(path)[:-4])
    try:
        mp.spawn(dp_bench.dp_worker,
                 args=(2, m, bs, res, 1, WARMUP, MEASURED, path, free_port(), WARM_SEC),
                 nprocs=2, join=True)
    finally:
        stop_clock_log(p, f)


In [12]:
# ---- Cell 12: run Experiment C (128x128) ----
RES2 = 128
for m in MODELS:
    for bs in (16, 32):
        bench_1gpu(m, bs, RES2)     # batch 16 = per-GPU reference for DP at global 32
    run_dp(m, 32, RES2)
    bench_mp(m, 32, RES2)


resnet18-1gpu-bs16-res128: median 22.6 ms | peak 283 MB | warm-up 2667 steps
resnet18-1gpu-bs32-res128: median 36.2 ms | peak 395 MB | warm-up 1670 steps


/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


resnet18-dp-bs32-res128: step 29.0 ms | exposed 7.0 ms | nccl 10.4 ms | warm-up 1207 steps
resnet18-mp-bs32-res128: step 28.9 ms | GPU0 17.8 | GPU1 11.1 | xfer 0.99 ms | warm-up 2048 steps
resnet34-1gpu-bs16-res128: median 38.8 ms | peak 424 MB | warm-up 1572 steps
resnet34-1gpu-bs32-res128: median 60.9 ms | peak 586 MB | warm-up 1010 steps


/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


resnet34-dp-bs32-res128: step 50.5 ms | exposed 12.7 ms | nccl 19.7 ms | warm-up 708 steps
resnet34-mp-bs32-res128: step 47.5 ms | GPU0 25.7 | GPU1 21.3 | xfer 1.00 ms | warm-up 1258 steps
resnet50-1gpu-bs16-res128: median 65.1 ms | peak 711 MB | warm-up 921 steps
resnet50-1gpu-bs32-res128: median 115.4 ms | peak 1160 MB | warm-up 525 steps


...

resnet50-dp-bs32-res128: step 75.0 ms | exposed 9.4 ms | nccl 22.7 ms | warm-up 436 steps
resnet50-mp-bs32-res128: step 91.1 ms | GPU0 46.9 | GPU1 40.7 | xfer 3.55 ms | warm-up 655 steps
vittiny-1gpu-bs16-res128: median 24.5 ms | peak 228 MB | warm-up 2525 steps
vittiny-1gpu-bs32-res128: median 43.3 ms | peak 393 MB | warm-up 1402 steps


/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


vittiny-dp-bs32-res128: step 32.7 ms | exposed 7.7 ms | nccl 5.1 ms | warm-up 1007 steps
vittiny-mp-bs32-res128: step 33.4 ms | GPU0 17.2 | GPU1 16.4 | xfer 0.47 ms | warm-up 1769 steps
vitbase-1gpu-bs16-res128: median 211.3 ms | peak 1338 MB | warm-up 279 steps
vitbase-1gpu-bs32-res128: median 400.1 ms | peak 1979 MB | warm-up 151 steps


/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


vitbase-dp-bs32-res128: step 236.4 ms | exposed 22.0 ms | nccl 79.3 ms | warm-up 135 steps
vitbase-mp-bs32-res128: step 359.9 ms | GPU0 177.6 | GPU1 173.0 | xfer 1.49 ms | warm-up 169 steps


## Experiment D: NCCL transport test
The all-reduce test twice: NCCL's default transport, then with direct P2P allowed on any topology (`NCCL_P2P_LEVEL=SYS`). Runs last, so a problem here can't affect the other results.

In [13]:
# ---- Cell 13: Experiment D ----
grad_mb = sorted(sum(p.numel() for p in build_model(m, 224).parameters()) * 4 / 2**20 for m in MODELS)
sizes = sorted(set([1, 4, 16, 64, 256] + grad_mb))
for name, env in [('default', {}), ('p2p_forced', {'NCCL_P2P_LEVEL': 'SYS'})]:
    path = f"{RESULTS_DIR}/allreduce-{name}.csv"
    if already_done(path): continue
    print(f"--- all-reduce with transport setting: {name}")
    try:
        mp.spawn(dp_bench.allreduce_worker, args=(2, sizes, 50, path, free_port(), env),
                 nprocs=2, join=True)
    except Exception as e:
        print(f"{name} failed: {type(e).__name__}: {e}")


--- all-reduce with transport setting: default
94ef6754dff1:440:440 [1] NCCL INFO ENV/Plugin: Could not find: libnccl-env.so
94ef6754dff1:440:440 [1] NCCL INFO cudaDriverVersion 13000
94ef6754dff1:440:440 [1] NCCL INFO NCCL version 2.31.2+cuda13.3
94ef6754dff1:440:440 [1] NCCL INFO NCCL git version unknown unknown
94ef6754dff1:440:440 [1] NCCL INFO NET/Plugin: Could not find: libnccl-net.so
94ef6754dff1:440:440 [1] NCCL INFO NET/IB : No device found.
94ef6754dff1:440:440 [1] NCCL INFO NET/IB : Using [RO]; OOB eth0:172.19.2.2<0>
94ef6754dff1:440:440 [1] NCCL INFO Failed to initialize NET plugin IB
94ef6754dff1:440:440 [1] NCCL INFO NET/Socket : Using [0]eth0:172.19.2.2<0>
94ef6754dff1:440:440 [1] NCCL INFO Initialized NET plugin Socket
94ef6754dff1:440:440 [1] NCCL INFO Assigned NET plugin Socket to comm
94ef6754dff1:440:440 [1] NCCL INFO RMA/Plugin: Could not find: libnccl-rma.so
94ef6754dff1:440:440 [1] NCCL INFO RMA/Plugin: Failed to initialize any plugin
94ef6754dff1:440:440 [1] NCC

In [14]:
# ---- Cell 14: summary of saved files ----
files = sorted(os.listdir(RESULTS_DIR))
res_files = [f for f in files if f.endswith('.csv') and 'clocks' not in f]
print(len(res_files), "result files:")
for f in res_files: print("  ", f)


36 result files:
   allreduce-default.csv
   allreduce-p2p_forced.csv
   resnet18-1gpu-bs16-res128.csv
   resnet18-1gpu-bs32-res128.csv
   resnet18-coldstart-bs32-res224.csv
   resnet18-dp-bs32-res128.csv
   resnet18-duty0-bs32-res224.csv
   resnet18-duty100-bs32-res224.csv
   resnet18-duty50-bs32-res224.csv
   resnet18-mp-bs32-res128.csv
   resnet34-1gpu-bs16-res128.csv
   resnet34-1gpu-bs32-res128.csv
   resnet34-dp-bs32-res128.csv
   resnet34-mp-bs32-res128.csv
   resnet50-1gpu-bs16-res128.csv
   resnet50-1gpu-bs32-res128.csv
   resnet50-coldstart-bs32-res224.csv
   resnet50-dp-bs32-res128.csv
   resnet50-duty0-bs32-res224.csv
   resnet50-duty100-bs32-res224.csv
   resnet50-duty50-bs32-res224.csv
   resnet50-mp-bs32-res128.csv
   vitbase-1gpu-bs16-res128.csv
   vitbase-1gpu-bs32-res128.csv
   vitbase-dp-bs32-res128.csv
   vitbase-duty0-bs16-res224.csv
   vitbase-duty100-bs16-res224.csv
   vitbase-duty50-bs16-res224.csv
   vitbase-mp-bs32-res128.csv
   vittiny-1gpu-bs16-res128.csv
  